# オホーツク海高気圧の見直し

同じ天気図を、**以前の答えを見ずに**もう一度判定します。

抜き取り200枚での結果は **κ = 0.464（中程度）／人間どうしのF1 = 0.500**。
判断が揺れており、しかも2回目の方が陽性を多く見つけました（10枚増・4枚減）。
見落としが相当数あると考えられるため、範囲を広げて付け直します。

## 始める前に、判定基準を文章にしてください

下のセルに書き留めてから始めると、途中で基準が揺れにくくなります。
κ 0.464 という値は、基準が言語化されていないことの表れでもあります。

- 結果は `data/review_okhotsk_full.csv` に追記されます。**元の labels.csv は変更しません。**
- 途中で閉じても、続きから再開できます。
- 迷ったら「わからない」を押してください。無理に決めると測定の意味が薄れます。
- **モデルの予測やERA5の数値は表示しません。** 見えているとそちらに引きずられ、
  「モデルの出力でモデルを評価する」ことになってしまうためです。

In [ ]:
import sys
from pathlib import Path

# リポジトリの場所を自動で探す(ノートブックをどこから開いても動くように)
here = Path.cwd()
repo = next((p for p in [here, *here.parents] if (p / "src" / "labels.py").exists()), None)
if repo is None:
    raise SystemExit("リポジトリのルートが見つかりません")
sys.path.insert(0, str(repo))

# 天気図の画像がある場所。環境に合わせて書き換えてください。
IMAGES_DIR = repo.parent / "weather-pattern-classification-data" / "processed"
LABELS_CSV = repo / "data" / "labels.csv"
OUT_CSV = repo / "data" / "review_okhotsk.csv"

print("画像:", IMAGES_DIR, "(あり)" if IMAGES_DIR.exists() else "(見つかりません)")
print("ラベル:", LABELS_CSV, "(あり)" if LABELS_CSV.exists() else "(見つかりません)")

## 判定基準（始める前に記入してください）

> オホーツク海高気圧を「あり」とするのは、
> 
> - オホーツク海（おおむね北緯45〜60度・東経135〜160度）に高気圧の中心があり、
> - かつ ……（ここに条件を書く。例: その高気圧の縁が北日本にかかり、等圧線から北東〜東寄りの風が北日本に入ると判断できる）
> 
> 迷う場合の扱い: ……

In [ ]:
from scripts.label_tool import run_binary_review_session

# months で対象の月を絞る。sample=None で、その範囲を全部見る。
#   [5, 6, 7, 8]      … 738枚 (現在の陽性の85%を含む / 約1.6時間)
#   range(4, 10)      … 1158枚 (96%を含む / 約2.6時間)  ← 推奨
#   None              … 2432枚 全期間 (約5.4時間)
#
# 範囲を狭めると、その外にある事例を最初から見に行かないことになる。
# 200枚の抜き取りでは、新たに「あり」とした10枚のうち4枚が5〜8月の外にあった。
run_binary_review_session(
    images_dir=IMAGES_DIR,
    labels_csv=LABELS_CSV,
    out_csv=repo / "data" / "review_okhotsk_full.csv",
    label="okhotsk_high",
    months=list(range(4, 10)),
    sample=None,
)

## 終わったら

下のセルで、1回目の答えと突き合わせます。

In [ ]:
!python -m scripts.compare_review --review {OUT_CSV} --labels {LABELS_CSV} --label okhotsk_high

---

## 3回目：基準を1つに決める

1回目（labels.csv）と2回目（見直し）で、陽性の数が 82 → 195 と2.4倍になりました。
1回目の陽性の87.8%は2回目にも引き継がれているので、判断がばらついたのではなく
**2回目の基準のほうが緩い**という状態です。

そこで「**一度でも『あり』とした天気図**」だけを集めて、もう一度判定します。
ここで採用した基準が、最終的なラベルになります。

- 対象は約205枚。30〜40分
- 2回目で落とした10枚も候補に含まれます
- どの回でも「なし」だった天気図は対象外です。
  **この作業で新たな見落としは拾えません**（基準を揃えるための工程です）
- 1〜2月の陽性3枚も候補に入るので、これで全期間の陽性候補が揃います

In [ ]:
from scripts.label_tool import positives_union, run_binary_review_session

candidates = positives_union(
    LABELS_CSV,
    [repo / "data" / "review_okhotsk.csv", repo / "data" / "review_okhotsk_full.csv"],
    label="okhotsk_high",
)
print(f"一度でも「あり」とした天気図: {len(candidates)}枚")

run_binary_review_session(
    images_dir=IMAGES_DIR,
    labels_csv=LABELS_CSV,
    out_csv=repo / "data" / "review_okhotsk_final.csv",
    label="okhotsk_high",
    filenames=candidates,
    sample=None,
)

終わったら、これで最終ラベルを書き出します（元の labels.csv は変更しません）。

```powershell
python -m scripts.compare_review --review data\review_okhotsk_final.csv --labels data\labels.csv --label okhotsk_high --apply data\labels_v2.csv
```